# Training Production-Ready LLMs — PyTorch Implementations from Scratch

Companion notebook to *Training Production-Ready Large Language Models (2025–2026)*.
Each cell implements a core concept using only `torch` and `torch.nn`.

**Sections:**
1. Shared Imports
2. Attention Mechanisms
3. Positional Encodings
4. Normalization and Activation
5. Training Objectives
6. Optimization
7. Parameter-Efficient Fine-Tuning (LoRA)
8. Alignment Objectives (DPO, SimPO, GRPO)
9. Knowledge Distillation
10. Inference — Speculative Decoding
11. Model Merging (Task Arithmetic, DARE, TIES, SLERP)
12. Continual Learning — EWC
13. Putting It All Together — Minimal Decoder + TinyGPT

## 1. Shared Imports

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

## 2. Attention Mechanisms

### 2.1 Scaled Dot-Product Attention

Implements $\mathrm{softmax}(QK^\top/\sqrt{d_k})\,V$ with an optional causal mask.

> **Formula:** $\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$

In [ ]:
def scaled_dot_product_attention(
    Q: torch.Tensor,   # (B, H, T, d_k)
    K: torch.Tensor,   # (B, H, S, d_k)
    V: torch.Tensor,   # (B, H, S, d_v)
    causal: bool = False,
) -> torch.Tensor:
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # (B,H,T,S)

    if causal:
        T, S = scores.shape[-2], scores.shape[-1]
        mask = torch.triu(torch.ones(T, S, device=Q.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))

    return F.softmax(scores, dim=-1) @ V  # (B,H,T,d_v)

### 2.2 Multi-Head Attention (MHA)

Splits $d_\mathrm{model}$ into $H$ independent heads, runs attention in parallel, then projects back.

> **Formula:** $\mathrm{MHA}(X) = \mathrm{Concat}(\mathrm{head}_1,\ldots,\mathrm{head}_H)\,W_O$, where $\mathrm{head}_i = \mathrm{Attention}(XW_Q^i, XW_K^i, XW_V^i)$

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.d_k = n_heads, d_model // n_heads
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)

    def _split_heads(self, x):          # (B,T,D) -> (B,H,T,d_k)
        B, T, _ = x.shape
        return x.view(B, T, self.h, self.d_k).transpose(1, 2)

    def forward(self, x, causal=True):
        Q = self._split_heads(self.Wq(x))
        K = self._split_heads(self.Wk(x))
        V = self._split_heads(self.Wv(x))
        out = scaled_dot_product_attention(Q, K, V, causal=causal)
        B, H, T, dk = out.shape
        out = out.transpose(1, 2).reshape(B, T, H * dk)
        return self.Wo(out)

### 2.3 Grouped Query Attention (GQA)

$G$ KV groups, each shared by $H/G$ query heads. Setting $G=1$ recovers MQA; $G=H$ recovers MHA.

> **Used in:** Llama 3, Mistral, Qwen, Gemma 2

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, n_kv_groups: int):
        super().__init__()
        assert n_heads % n_kv_groups == 0
        self.h, self.g = n_heads, n_kv_groups
        self.d_k = d_model // n_heads
        self.Wq = nn.Linear(d_model, n_heads     * self.d_k, bias=False)
        self.Wk = nn.Linear(d_model, n_kv_groups * self.d_k, bias=False)
        self.Wv = nn.Linear(d_model, n_kv_groups * self.d_k, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, causal=True):
        B, T, _ = x.shape
        q = self.Wq(x).view(B, T, self.h, self.d_k).transpose(1, 2)
        k = self.Wk(x).view(B, T, self.g, self.d_k).transpose(1, 2)
        v = self.Wv(x).view(B, T, self.g, self.d_k).transpose(1, 2)

        # Broadcast KV groups across query heads
        reps = self.h // self.g
        k = k.repeat_interleave(reps, dim=1)  # (B, H, T, d_k)
        v = v.repeat_interleave(reps, dim=1)

        out = scaled_dot_product_attention(q, k, v, causal=causal)
        out = out.transpose(1, 2).reshape(B, T, -1)
        return self.Wo(out)

## 3. Positional Encodings

### 3.1 RoPE (Rotary Position Embedding)

Applies a position-dependent rotation to Q and K so that $q_t \cdot k_s$ depends only on $(t-s)$.

> **Formula:** $q_t' = q_t e^{i\theta_t}$, where $\theta_{t,2i} = t \cdot b^{-2i/d}$

In [ ]:
def precompute_rope_freqs(d_k: int, max_seq: int, base: float = 10000.0):
    # theta_i = base^{-2i/d}  for i = 0 .. d/2-1
    inv_freq = 1.0 / (base ** (torch.arange(0, d_k, 2).float() / d_k))
    t = torch.arange(max_seq).float()
    freqs = torch.outer(t, inv_freq)            # (max_seq, d/2)
    return torch.cat([freqs, freqs], dim=-1)    # (max_seq, d)

def rotate_half(x):
    half = x.shape[-1] // 2
    return torch.cat([-x[..., half:], x[..., :half]], dim=-1)

def apply_rope(x, freqs):
    # x: (B, H, T, d_k)  freqs: (T, d_k)
    freqs = freqs[:x.shape[2]].unsqueeze(0).unsqueeze(0)
    return x * freqs.cos() + rotate_half(x) * freqs.sin()

### 3.2 ALiBi (Attention with Linear Biases)

Subtracts a linear penalty $m_h \cdot (i-j)$ from attention logits. No learned parameters; extrapolates to unseen lengths.

> **Formula:** $\mathrm{Attention}_{ij} = \mathrm{softmax}(q_i k_j^\top / \sqrt{d} - m_h \cdot |i-j|)$

In [ ]:
def alibi_bias(n_heads: int, seq_len: int) -> torch.Tensor:
    # Head slopes: geometric sequence  2^{-8/H}, 2^{-16/H}, ...
    slopes = 2 ** (-8 * torch.arange(1, n_heads + 1) / n_heads)   # (H,)
    pos    = torch.arange(seq_len).unsqueeze(0) \
           - torch.arange(seq_len).unsqueeze(1)                    # (T, T)
    # bias[h,i,j] = -slopes[h] * max(0, i-j)
    bias = -slopes.view(-1, 1, 1) * pos.abs().unsqueeze(0).float()
    return bias.tril()   # (H, T, T) -- lower triangular, causal

## 4. Normalization and Activation

### 4.1 RMSNorm

> **Formula:** $\mathrm{RMSNorm}(x) = \gamma \odot x / (\|x\|_\mathrm{RMS} + \epsilon)$, where $\|x\|_\mathrm{RMS} = \sqrt{\frac{1}{d}\sum_i x_i^2}$

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-8):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return self.gamma * x / rms

### 4.2 SwiGLU Feed-Forward Block

> **Formula:** $\mathrm{FFN}(x) = (\mathrm{Swish}(xW_1) \odot xW_2)\,W_3$, where $\mathrm{Swish}(x) = x \cdot \sigma(x)$

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.W1 = nn.Linear(d_model, d_ff, bias=False)  # gate
        self.W2 = nn.Linear(d_model, d_ff, bias=False)  # value
        self.W3 = nn.Linear(d_ff,    d_model, bias=False)

    def forward(self, x):
        return self.W3(F.silu(self.W1(x)) * self.W2(x))
        # F.silu(x) = x * sigmoid(x)  is Swish

## 5. Training Objectives

### 5.1 Causal LM Loss with Token Masking

SFT masks prompt tokens so loss is computed only on assistant responses.

> **Formula:** $\mathcal{L}_{\mathrm{NTP}} = -\frac{1}{|\mathcal{R}|}\sum_{t \in \mathcal{R}} \log p_\theta(x_t \mid x_{<t})$

In [ ]:
def causal_lm_loss(logits, labels):
    """
    logits : (B, T, V)   raw model output
    labels : (B, T)      token ids; -100 marks tokens to ignore
    """
    # Shift: predict token t from tokens 0..t-1
    shift_logits = logits[:, :-1].contiguous().view(-1, logits.size(-1))
    shift_labels = labels[:, 1:].contiguous().view(-1)
    return F.cross_entropy(shift_logits, shift_labels, ignore_index=-100)

def perplexity(model, input_ids):
    with torch.no_grad():
        logits = model(input_ids).logits
    loss = causal_lm_loss(logits, input_ids)
    return loss.exp().item()

## 6. Optimization

### 6.1 AdamW — Manual Update Step

> **Formula:** $\theta \leftarrow (1 - \eta\lambda)\theta - \eta \hat{m}/(\sqrt{\hat{v}} + \epsilon)$

Weight decay is applied **before** the adaptive gradient step (decoupled).

In [ ]:
def adamw_step(params, grads, m, v, t,
               lr=3e-4, b1=0.9, b2=0.95, eps=1e-8, wd=0.1):
    t += 1
    for p, g, m_i, v_i in zip(params, grads, m, v):
        m_i.mul_(b1).add_(g, alpha=1 - b1)            # 1st moment
        v_i.mul_(b2).addcmul_(g, g, value=1 - b2)     # 2nd moment
        m_hat = m_i / (1 - b1 ** t)
        v_hat = v_i / (1 - b2 ** t)
        # Decoupled weight decay applied BEFORE adaptive step
        p.mul_(1 - lr * wd)
        p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)
    return t

### 6.2 Cosine LR Schedule with Linear Warmup

> **Formula:** $\eta(t) = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\!\left(\frac{t - t_{\mathrm{warmup}}}{T - t_{\mathrm{warmup}}}\pi\right)\right)$

In [ ]:
def cosine_lr(step: int, total_steps: int,
              lr_max: float, lr_min: float,
              warmup_steps: int) -> float:
    if step < warmup_steps:
        return lr_max * step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * progress))

# Example: visualise the schedule
TOTAL_STEPS, WARMUP_STEPS = 10000, 500
lrs = [cosine_lr(s, TOTAL_STEPS, 3e-4, 3e-5, WARMUP_STEPS) for s in range(TOTAL_STEPS)]
print(f"Peak LR: {max(lrs):.2e}  |  Final LR: {lrs[-1]:.2e}")

## 7. Parameter-Efficient Fine-Tuning

### 7.1 LoRA Linear Layer

Adds a low-rank branch $\Delta W = BA$ alongside a frozen base weight.

> **Formula:** $h = W_0 x + \frac{\alpha}{r} B A x$, where $A \in \mathbb{R}^{r \times d_{\mathrm{in}}}$, $B \in \mathbb{R}^{d_{\mathrm{out}} \times r}$, $B$ initialized to 0

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int = 16, alpha: float = 32.0):
        super().__init__()
        d_out, d_in = base.weight.shape
        self.base   = base
        self.base.weight.requires_grad_(False)   # freeze base
        self.A      = nn.Parameter(torch.randn(rank, d_in)  * 0.01)
        self.B      = nn.Parameter(torch.zeros(d_out, rank))
        self.scale  = alpha / rank               # alpha/r scaling

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale

    def merge(self):
        """Absorb LoRA into base weight (zero inference overhead)."""
        with torch.no_grad():
            self.base.weight.add_(self.scale * self.B @ self.A)
        del self.A, self.B

## 8. Alignment Objectives

### 8.1 DPO Loss (Direct Preference Optimization)

> **Formula:** $\mathcal{L}_{\mathrm{DPO}} = -\mathbb{E}\left[\log\sigma\!\left(\beta\left(\log\frac{\pi_\theta(y_w|x)}{\pi_{\mathrm{ref}}(y_w|x)} - \log\frac{\pi_\theta(y_l|x)}{\pi_{\mathrm{ref}}(y_l|x)}\right)\right)\right]$

In [ ]:
def log_prob_seq(model, input_ids, labels):
    """Sum of log-probs over non-masked label tokens."""
    logits = model(input_ids).logits[:, :-1]
    tgt    = labels[:, 1:]
    lp     = F.log_softmax(logits, dim=-1)
    mask   = (tgt != -100)
    return (lp.gather(-1, tgt.clamp(min=0).unsqueeze(-1)).squeeze(-1)
              * mask).sum(-1)

def dpo_loss(policy, ref, x_w, x_l, beta=0.1):
    """
    x_w / x_l : tokenised (prompt + chosen/rejected) tensors (B, T)
    Labels equal input_ids but with prompt tokens set to -100.
    """
    pi_w  = log_prob_seq(policy, x_w, x_w)
    pi_l  = log_prob_seq(policy, x_l, x_l)
    ref_w = log_prob_seq(ref,    x_w, x_w)
    ref_l = log_prob_seq(ref,    x_l, x_l)
    reward_margin = beta * ((pi_w - ref_w) - (pi_l - ref_l))
    return -F.logsigmoid(reward_margin).mean()

### 8.2 SimPO Loss

No reference model required; uses length-normalised log-prob as reward.

> **Formula:** $\mathcal{L}_{\mathrm{SimPO}} = -\mathbb{E}\left[\log\sigma\!\left(\beta\left(\frac{1}{|y_w|}\log\pi_\theta(y_w|x) - \frac{1}{|y_l|}\log\pi_\theta(y_l|x)\right) - \gamma\right)\right]$

In [ ]:
def avg_log_prob(model, input_ids, labels):
    """Average log-prob per non-masked token."""
    logits = model(input_ids).logits[:, :-1]
    tgt    = labels[:, 1:]
    lp     = F.log_softmax(logits, dim=-1)
    mask   = (tgt != -100).float()
    token_lp = lp.gather(-1, tgt.clamp(min=0).unsqueeze(-1)).squeeze(-1)
    return (token_lp * mask).sum(-1) / mask.sum(-1).clamp(min=1)

def simpo_loss(model, x_w, x_l, beta=2.0, gamma=0.5):
    r_w = avg_log_prob(model, x_w, x_w)
    r_l = avg_log_prob(model, x_l, x_l)
    return -F.logsigmoid(beta * (r_w - r_l) - gamma).mean()

### 8.3 GRPO Training Step (Group Relative Policy Optimization)

Samples $G$ responses, normalises rewards within the group, clips the ratio.

> **Formula:** $\mathcal{L}_{\mathrm{GRPO}} = -\frac{1}{G}\sum_{i=1}^{G}\left[\min\!\left(r_i(\theta)\hat{A}_i,\ \mathrm{clip}(r_i(\theta), 1\pm\epsilon)\hat{A}_i\right) - \beta\,\mathrm{KL}(\pi_\theta \| \pi_{\mathrm{ref}})\right]$

In [ ]:
def grpo_step(policy, ref, prompt_ids, reward_fn,
              G=8, beta=0.001, eps=0.2):
    """
    reward_fn(completions) -> tensor (G,)  verifiable reward per completion
    """
    # 1. Sample G completions from the old (frozen) policy
    with torch.no_grad():
        completions = [policy.generate(prompt_ids) for _ in range(G)]
        rewards     = reward_fn(completions)                    # (G,)
        old_logp    = torch.stack(
            [log_prob_seq(policy, c, c) for c in completions]) # (G,)

    # 2. Group-normalise advantages
    A_hat = (rewards - rewards.mean()) / (rewards.std() + 1e-8)

    # 3. GRPO objective
    total_loss = 0.0
    for i, comp in enumerate(completions):
        logp     = log_prob_seq(policy, comp, comp)
        ratio    = (logp - old_logp[i]).exp()
        clipped  = ratio.clamp(1 - eps, 1 + eps)
        pg_loss  = -torch.min(ratio * A_hat[i], clipped * A_hat[i])

        # KL penalty vs reference
        kl = log_prob_seq(policy, comp, comp) \
           - log_prob_seq(ref,    comp, comp)
        total_loss += pg_loss + beta * kl

    return total_loss / G

## 9. Knowledge Distillation

### 9.1 Forward KL Distillation Loss

Combined cross-entropy on hard labels and KL divergence on soft teacher distribution.

> **Formula:** $\mathcal{L} = \alpha\,\mathcal{L}_{\mathrm{CE}} + (1-\alpha)\,T^2\,\mathrm{KL}(p_T^\tau \| p_S^\tau)$

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels,
                      alpha=0.5, temperature=2.0):
    """
    student_logits, teacher_logits : (B, T, V)
    labels : (B, T)  with -100 for masked positions
    """
    B, T, V = student_logits.shape
    s = student_logits.view(-1, V)
    t = teacher_logits.view(-1, V)

    ce_loss = F.cross_entropy(s, labels.view(-1), ignore_index=-100)

    soft_t  = F.softmax(t  / temperature, dim=-1)
    soft_s  = F.log_softmax(s / temperature, dim=-1)
    kd_loss = F.kl_div(soft_s, soft_t, reduction='batchmean') \
              * (temperature ** 2)

    return alpha * ce_loss + (1 - alpha) * kd_loss

## 10. Inference — Speculative Decoding

Draft model proposes $k$ tokens; target accepts each with probability $\min(1,\,p_{\mathrm{target}}/p_{\mathrm{draft}})$.

> **Speedup:** $\mathbb{E}[\text{tokens per step}] = k \cdot \alpha + 1$ where $\alpha$ is the average acceptance rate.

In [ ]:
@torch.no_grad()
def speculative_decode(target, draft, input_ids, max_new=128, k=4):
    """Returns completed token sequence."""
    ids = input_ids.clone()

    while ids.shape[1] < input_ids.shape[1] + max_new:
        # Step 1: draft proposes k tokens
        draft_ids = ids.clone()
        draft_probs = []
        for _ in range(k):
            d_logits = draft(draft_ids).logits[:, -1]
            p_draft  = F.softmax(d_logits, dim=-1)
            tok      = torch.multinomial(p_draft, 1)
            draft_probs.append(p_draft[0, tok[0, 0]].item())
            draft_ids = torch.cat([draft_ids, tok], dim=1)

        # Step 2: target scores the entire draft in one forward pass
        t_logits = target(draft_ids).logits                    # (1, T+k, V)
        accepted = 0
        for j in range(k):
            p_tgt   = F.softmax(t_logits[0, ids.shape[1]+j-1], dim=-1)
            tok_j   = draft_ids[0, ids.shape[1] + j].item()
            alpha   = min(1.0, p_tgt[tok_j].item() / (draft_probs[j] + 1e-9))
            if torch.rand(1).item() < alpha:
                accepted += 1
            else:            # rejection: sample correction token
                p_corr = (p_tgt - F.softmax(t_logits[0, ids.shape[1]+j-1], dim=-1))
                p_corr = p_corr.clamp(min=0)
                p_corr /= p_corr.sum()
                tok_j  = torch.multinomial(p_corr, 1).item()
                break

        ids = torch.cat([ids, draft_ids[:, ids.shape[1]:ids.shape[1]+accepted+1]], dim=1)

    return ids

## 11. Model Merging

### 11.1 Task Arithmetic

> **Formula:** $\theta_{\mathrm{merged}} = \theta_0 + \sum_i \lambda_i \tau_i$, where $\tau_i = \theta_{\mathrm{ft},i} - \theta_0$

In [ ]:
def task_vector(base_sd, ft_sd):
    """Compute tau = theta_ft - theta_base for matching state-dicts."""
    return {k: ft_sd[k].float() - base_sd[k].float() for k in base_sd}

def apply_vectors(base_sd, vectors, lambdas):
    """theta_merged = theta_base + sum_i lambda_i * tau_i"""
    merged = {k: base_sd[k].float().clone() for k in base_sd}
    for tau, lam in zip(vectors, lambdas):
        for k in merged:
            merged[k].add_(tau[k], alpha=lam)
    return {k: v.to(base_sd[k].dtype) for k, v in merged.items()}

### 11.2 DARE (Drop And REscale)

> **Formula:** $\hat{\tau}_i = \frac{m_i \odot \tau_i}{1 - p}$, where $m_i \sim \mathrm{Bernoulli}(1-p)$

In [ ]:
def dare(tau: dict, drop_rate: float = 0.3) -> dict:
    """Zero drop_rate fraction of delta params; rescale survivors."""
    out = {}
    for k, v in tau.items():
        mask = torch.bernoulli(torch.full_like(v, 1 - drop_rate))
        out[k] = (mask * v) / (1 - drop_rate)
    return out

### 11.3 TIES-Merging (Trim, Elect Sign, Merge)

> **Steps:** (1) Trim small-magnitude deltas, (2) Elect majority sign per parameter, (3) Average only models whose sign agrees

In [ ]:
def ties_merge(base_sd, ft_sds, lambdas, density=0.7):
    """
    ft_sds  : list of fine-tuned state-dicts
    lambdas : per-model merge coefficients
    density : fraction of delta params to keep (1-trim rate)
    """
    taus = [task_vector(base_sd, ft) for ft in ft_sds]

    # Step 1 -- Trim: keep top-density fraction by magnitude
    trimmed = []
    for tau in taus:
        t = {}
        for k, v in tau.items():
            thresh = v.abs().flatten().quantile(1 - density)
            t[k]   = v * (v.abs() >= thresh)
        trimmed.append(t)

    merged = {}
    for k in base_sd:
        stack = torch.stack([t[k].float() for t in trimmed], dim=0)  # (M,...)

        # Step 2 -- Elect majority sign
        sign = torch.sign(stack.sum(dim=0))  # +1 / -1 / 0

        # Step 3 -- Average only models whose sign agrees
        agree = (torch.sign(stack) == sign.unsqueeze(0)).float()
        denom = agree.sum(dim=0).clamp(min=1)
        delta = (stack * agree).sum(dim=0) / denom

        merged[k] = base_sd[k].float() + delta * sum(lambdas)

    return {k: v.to(base_sd[k].dtype) for k, v in merged.items()}

### 11.4 SLERP (Spherical Linear Interpolation)

Interpolates two weight tensors along the great-circle arc.

> **Formula:** $\mathrm{SLERP}(\theta_A, \theta_B, t) = \frac{\sin((1-t)\Omega)}{\sin\Omega}\theta_A + \frac{\sin(t\Omega)}{\sin\Omega}\theta_B$, where $\Omega = \arccos(\hat{\theta}_A \cdot \hat{\theta}_B)$

In [ ]:
def slerp(w_a: torch.Tensor, w_b: torch.Tensor, t: float) -> torch.Tensor:
    a = w_a.float().flatten()
    b = w_b.float().flatten()
    cos_omega = (a / a.norm()).dot(b / b.norm()).clamp(-1, 1)
    omega = cos_omega.acos()
    if omega.abs() < 1e-6:            # nearly parallel -- fall back to lerp
        return ((1 - t) * w_a + t * w_b).to(w_a.dtype)
    out = (math.sin((1 - t) * omega) * a
         + math.sin(t       * omega) * b) / math.sin(omega)
    return out.reshape(w_a.shape).to(w_a.dtype)

def slerp_state_dict(sd_a, sd_b, t=0.5):
    return {k: slerp(sd_a[k], sd_b[k], t) for k in sd_a}

## 12. Continual Learning — EWC (Elastic Weight Consolidation)

Computes the diagonal Fisher information matrix on task A, then adds a penalty when training on task B.

> **Formula:** $\mathcal{L}_B(\theta) = \mathcal{L}_B^{\mathrm{task}}(\theta) + \frac{\lambda}{2}\sum_i F_i(\theta_i - \theta_{A,i}^*)^2$

In [ ]:
class EWC:
    def __init__(self, model: nn.Module, dataset, lam: float = 1000.0):
        self.lam    = lam
        self.params = {n: p.clone().detach()
                       for n, p in model.named_parameters() if p.requires_grad}
        self.fisher = self._compute_fisher(model, dataset)

    def _compute_fisher(self, model, dataset):
        fisher = {n: torch.zeros_like(p)
                  for n, p in model.named_parameters() if p.requires_grad}
        model.eval()
        for x, y in dataset:
            model.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            for n, p in model.named_parameters():
                if p.requires_grad and p.grad is not None:
                    fisher[n] += p.grad.pow(2)
        n = len(dataset)
        return {n: f / n for n, f in fisher.items()}

    def penalty(self, model: nn.Module) -> torch.Tensor:
        loss = torch.tensor(0.0, device=next(model.parameters()).device)
        for n, p in model.named_parameters():
            if n in self.fisher:
                loss += (self.fisher[n] * (p - self.params[n]).pow(2)).sum()
        return 0.5 * self.lam * loss

# Training loop on task B:
#   total_loss = task_b_loss + ewc.penalty(model)

## 13. Putting It All Together — Minimal Decoder Block

A single transformer decoder layer assembling the components above, plus a minimal `TinyGPT` model.

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model=512, n_heads=8, n_kv=2, d_ff=2048):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn  = GroupedQueryAttention(d_model, n_heads, n_kv)
        self.norm2 = RMSNorm(d_model)
        self.ffn   = SwiGLU(d_model, d_ff)

    def forward(self, x, rope_freqs):
        # Pre-norm + residual connection (standard in 2025 models)
        h = self.norm1(x)
        # Inject RoPE into Q and K inside attention
        # (real impl patches Wq/Wk output; shown simplified here)
        x = x + self.attn(h, causal=True)
        x = x + self.ffn(self.norm2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab=32000, d=512, heads=8, kv=2, ff=2048,
                 layers=6, max_seq=2048):
        super().__init__()
        self.embed   = nn.Embedding(vocab, d)
        self.layers  = nn.ModuleList(
            [DecoderLayer(d, heads, kv, ff) for _ in range(layers)])
        self.norm    = RMSNorm(d)
        self.head    = nn.Linear(d, vocab, bias=False)
        self.head.weight = self.embed.weight   # weight tying
        self.freqs   = precompute_rope_freqs(d // heads, max_seq)

    def forward(self, idx):
        x = self.embed(idx)
        for layer in self.layers:
            x = layer(x, self.freqs)
        return self.head(self.norm(x))        # (B, T, vocab)

In [ ]:
# Smoke test -- instantiate TinyGPT and run a forward pass
model = TinyGPT(vocab=32000, d=512, heads=8, kv=2, ff=2048, layers=6)
total_params = sum(p.numel() for p in model.parameters())
print(f"TinyGPT parameters: {total_params / 1e6:.1f}M")

dummy_input = torch.randint(0, 32000, (2, 64))  # batch=2, seq=64
logits = model(dummy_input)
print(f"Output shape: {logits.shape}")           # (2, 64, 32000)

loss = causal_lm_loss(logits, dummy_input)
print(f"Cross-entropy loss: {loss.item():.4f}")  # ~10.37 = log(32000)

## 14. Data Loading — Sample Dataset

A simple synthetic dataset generator for initial verification.

In [ ]:
from torch.utils.data import IterableDataset, DataLoader

class SampleDataset(IterableDataset):
    def __init__(self, vocab_size: int, seq_len: int):
        self.vocab_size = vocab_size
        self.seq_len = seq_len

    def __iter__(self):
        while True:
            # Generate random tokens for verification
            yield torch.randint(0, self.vocab_size, (self.seq_len,))

def get_dataloader(vocab_size: int, seq_len: int, batch_size: int):
    dataset = SampleDataset(vocab_size, seq_len)
    return DataLoader(dataset, batch_size=batch_size)

## 15. Training Loop — End-to-End Implementation

Integrates the `TinyGPT` model with the `AdamW` optimizer and `cosine_lr` schedule described in the book.

In [ ]:
def train_tiny_gpt(model, config):
    device = config.get('device', 'cpu')
    model.to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    dataloader = get_dataloader(config['vocab'], config['max_seq'], config['batch_size'])
    
    model.train()
    for step in range(config['max_steps']):
        # 1. Update learning rate per cosine schedule
        lr = cosine_lr(step, config['max_steps'], config['lr'], config['lr'] * 0.1, config['warmup'])
        for pg in optimizer.param_groups: pg['lr'] = lr

        # 2. Forward pass
        x = next(iter(dataloader)).to(device)
        y = x.clone() # Next-token prediction: labels are the inputs shifted
        
        logits = model(x)
        loss = causal_lm_loss(logits, y)
        
        # 3. Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if step % 10 == 0:
            print(f"Step {step:03d} | Loss: {loss.item():.4f} | LR: {lr:.2e}")

    print("Training complete.")

## 16. Verification Run

Running a minimal training loop on a scaled-down `TinyGPT` instance.

In [ ]:
train_config = {
    'vocab': 32000,
    'd': 128,
    'heads': 4,
    'kv': 2,
    'ff': 512,
    'layers': 2,
    'max_seq': 64,
    'lr': 1e-3,
    'weight_decay': 0.1,
    'batch_size': 16,
    'max_steps': 50,
    'warmup': 10,
    'device': 'cpu' # 'cuda' if torch.cuda.is_available() else 'cpu'
}

test_model = TinyGPT(
    vocab=train_config['vocab'], 
    d=train_config['d'], 
    heads=train_config['heads'], 
    kv=train_config['kv'], 
    ff=train_config['ff'], 
    layers=train_config['layers'], 
    max_seq=train_config['max_seq']
)

train_tiny_gpt(test_model, train_config)